# 🌊 Exploration des Données INRH & Détection d'Anomalies
**Mémoire de Master :** Chaimae Anbi  
**Sujet :** Détection d'anomalies et scoring de qualité dans un Data Lake scientifique  

Ce notebook permet d'explorer les 4 flux de données (VMS, LOGBOOK, SALES, ENV) et d'expérimenter les règles de qualité.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ajouter le dossier racine pour importer nos modules
sys.path.append('..')
from generator.generate_all import generate_vms_data, generate_env_data
from quality_engine.dimensions.completeness import evaluate_dataset_completeness
from quality_engine.dimensions.validity import check_maritime_coordinates, check_range_validity
from quality_engine.anomaly_detector import detect_outliers_zscore, detect_outliers_iqr
from quality_engine.scoring import compute_global_score

print('✅ Environnement et bibliothèques chargés avec succès !')

## 1. Génération et Inspection d'un échantillon VMS

In [ ]:
df_vms = generate_vms_data(n_rows=200)
df_vms.head()

## 2. Détection d'anomalies statistiques sur la vitesse (IQR)

In [ ]:
iqr_results = detect_outliers_iqr(df_vms, 'speed_knots')
print('Borne supérieure normale :', iqr_results['upper_bound'], 'nœuds')
print('Nombre d\'anomalies détectées :', iqr_results['outliers_count'])
df_vms.loc[iqr_results['indices']]

## 3. Calcul du Score de Qualité DAMA

In [ ]:
comp = evaluate_dataset_completeness(df_vms)
geo = check_maritime_coordinates(df_vms)
speed = check_range_validity(df_vms, 'speed_knots', 0, 35)

score_res = compute_global_score(
    completeness_score=comp['score'],
    validity_score=(geo['score'] + speed['score']) / 2,
    uniqueness_score=97.0,
    consistency_score=100.0
)
print(f"🎯 Score Global : {score_res['global_score']}% -> {score_res['grade']}")